<a href="https://colab.research.google.com/github/Diana102001/Ranking-medical-files-pipeline/blob/main/BagOfWords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('/content/train.xlsx')
df

,ID,Document
0,MED-10,statin breast cancer survival nationwide cohor...
1,MED-14,statin diagnosis breast cancer survival popula...
2,MED-118,alkylphenols human milk relations dietary habi...
3,MED-301,methylmercury potential environmental risk fac...
4,MED-306,sensitivity continuous performance test cpt ag...
...,...,...
3607,MED-5367,relationship plasma carotenoids depressive sym...
3608,MED-5368,suicide mortality relation dietary intake num ...
3609,MED-5369,suicide mortality european union pubmed ncbi a...
3610,MED-5370,long chain omega num fatty acids intake fish c...


In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from textblob import TextBlob, Word

df['2nd_column_lemmatized'] = df['Document'].apply(lambda x: ' '.join([Word(word).lemmatize() for word in x.split()]))
df.head()

,ID,Document,2nd_column_lemmatized
0,MED-10,statin breast cancer survival nationwide cohor...,statin breast cancer survival nationwide cohor...
1,MED-14,statin diagnosis breast cancer survival popula...,statin diagnosis breast cancer survival popula...
2,MED-118,alkylphenols human milk relations dietary habi...,alkylphenols human milk relation dietary habit...
3,MED-301,methylmercury potential environmental risk fac...,methylmercury potential environmental risk fac...
4,MED-306,sensitivity continuous performance test cpt ag...,sensitivity continuous performance test cpt ag...


In [ ]:
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer("english")

# Apply the Snowball stemmer to the 2nd column of the dataset
df['2nd_column_stemmed'] = df['Document'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

# Print the updated dataset
df.head()

,ID,Document,2nd_column_lemmatized,2nd_column_stemmed
0,MED-10,statin breast cancer survival nationwide cohor...,statin breast cancer survival nationwide cohor...,statin breast cancer surviv nationwid cohort s...
1,MED-14,statin diagnosis breast cancer survival popula...,statin diagnosis breast cancer survival popula...,statin diagnosi breast cancer surviv populatio...
2,MED-118,alkylphenols human milk relations dietary habi...,alkylphenols human milk relation dietary habit...,alkylphenol human milk relat dietari habit cen...
3,MED-301,methylmercury potential environmental risk fac...,methylmercury potential environmental risk fac...,methylmercuri potenti environment risk factor ...
4,MED-306,sensitivity continuous performance test cpt ag...,sensitivity continuous performance test cpt ag...,sensit continu perform test cpt age num year d...


In [ ]:
for word in lexicon:
    df['2nd_column_stemmed_after_deletion'] = df['2nd_column_stemmed'].str.replace(word, '')

In [ ]:
df.head()

In [ ]:
queries = pd.read_csv('/content/queries.csv')
queries

,PID,PLAIN
0,PLAIN-10,how contaminated are our children ?
1,PLAIN-100,cancer and the animal-to-plant protein ratio
2,PLAIN-1000,daniel fast
3,PLAIN-1002,dark chocolate
4,PLAIN-1003,dark meat
...,...,...
2589,PLAIN-993,cyclamate
2590,PLAIN-994,cycling
2591,PLAIN-995,cysticercosis
2592,PLAIN-998,dairy


In [ ]:
queries['lemmatized']=queries['PLAIN'].apply(lambda x: ' '.join([Word(word).lemmatize() for word in x.split()]))
queries

,PID,PLAIN,lemmatized
0,PLAIN-10,how contaminated are our children ?,how contaminated are our child ?
1,PLAIN-100,cancer and the animal-to-plant protein ratio,cancer and the animal-to-plant protein ratio
2,PLAIN-1000,daniel fast,daniel fast
3,PLAIN-1002,dark chocolate,dark chocolate
4,PLAIN-1003,dark meat,dark meat
...,...,...,...
2589,PLAIN-993,cyclamate,cyclamate
2590,PLAIN-994,cycling,cycling
2591,PLAIN-995,cysticercosis,cysticercosis
2592,PLAIN-998,dairy,dairy


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def get_top_5(dictionary):
    # Sort the dictionary items based on values in descending order
    sorted_items = sorted(dictionary.items(), key=lambda x: x[1], reverse=True)

    # Return the top 5 items
    return sorted_items[0:5]

# top_5 = get_top_5(scores)
# print(top_5)

In [ ]:
import nltk
from nltk import word_tokenize
from nltk.util import ngrams
from collections import Counter

In [ ]:
qlist=[]
docId=[]
queriesList=queries['lemmatized']
documents=df['2nd_column_lemmatized']
docIds=df['ID']
scores={}
i=0
iteration=0

for q in queriesList:
    query=nltk.word_tokenize(q)
    # query = queriesList[i]  # Assuming queriesList is already a list of lemmatized queries
    if(len(query)>1):

      query_bigrams = list(ngrams(query, 2))
           # Generate bigrams for the query
      # print(query_bigrams)
      for d in documents:
          score = 0
          lemmas = Counter(nltk.word_tokenize(d))
          document_bigrams = list(ngrams(lemmas, 2))  # Generate bigrams for the document
          # print(document_bigrams)
          for bg in query_bigrams:
              if bg in document_bigrams:
                  # Check if the bigram is present in the document
                  print("OK")
                  # print(bg)
                  # print("start")
                  score += 1
          id = docIds[i]
          scores[id] = score
          print(score)
          i += 1
      top_5_scores = get_top_5(scores)  # Assuming get_top_5 function returns the top 5 scores
      sc = '|'.join(map(str, [top[1] for top in top_5_scores]))
      result_string = '|'.join([top[0] for top in top_5_scores])
      print(result_string)
      qlist.append(sc)
      docId.append(result_string)
      i = 0
      print(iteration)
      iteration += 1
      print(sc)
      print("in gram")
      #if(iteration>5): break
    else:
        for d in documents:
          score=0
          lemmas = Counter(nltk.word_tokenize(d))
  # print (lemmas)
          for t in query:
    #  print(t)
    # print(lemmas.get(t))
            if(lemmas.get(t)):
              score=score+lemmas.get(t)
        id=docIds[i]
        scores[id]=score
        i=i+1
    sc = '|'.join(map(str, [top[1] for top in get_top_5(scores)]))
    id=result_string = '|'.join([top[0] for top in get_top_5(scores)])
    qlist.append(sc)
    docId.append(id)
    i=0
    print(iteration)
    print(sc)
    print("in morph")
    iteration=iteration+1
  #if(iteration>5): break



Streaming output truncated to the last 5000 lines.
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
OK
1
0
OK
1
0
0
OK
1
0
0


In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.corpus import wordnet

In [ ]:
# qlist=[]
# docId=[]
# queriesList=queries['lemmatized']
# documents=df['2nd_column_lemmatized']
# docIds=df['ID']
# scores={}
# i=0
# iteration=0
# wnl = WordNetLemmatizer()  # Initialize the WordNetLemmatizer

# for q in queriesList:
#     query=nltk.word_tokenize(q)
#     query_synsets = [wn.synsets(word) for word in query]  # Get WordNet synsets for each word in the query

#     for d in documents:
#         score = 0
#         lemmas = Counter(nltk.word_tokenize(d))

#         for synset_list in query_synsets:
#             for synset in synset_list:
#                 for lemma in synset.lemma_names():
#                     if lemmas.get(lemma):
#                         score += lemmas.get(lemma)

#         id = docIds[i]
#         scores[id] = score
#         i += 1

#     top_5_scores = get_top_5(scores)  # Assuming get_top_5 function returns the top 5 scores
#     sc = '|'.join(map(str, [top[1] for top in top_5_scores]))
#     result_string = '|'.join([top[0] for top in top_5_scores])
#     qlist.append(sc)
#     docId.append(result_string)
#     i = 0
#     print(iteration)
#     print(sc)
#     print(result_string)
#     iteration += 1
#     #if(iteration>5): break

In [ ]:
queries.to_csv('/content/drive/MyDrive/NLP_Dian/queries2Gram.csv', index=False)

In [ ]:
queries

In [ ]:
#add scores as column
queries['scores']=qlist
queries['docId']=docId
queries
#print(scores)

import numpy as np
def Thresholding(value):
  if value < 0.1: return 0
  elif value < 0.3: return 1
  elif value <0.7: return 2
  else : return 3

#normalize
import numpy as np

result_list=[]
for q in qlist:
  res_tem=q.split('|')
  res_tem_int=[int(element) for element in res_tem]
  result_list =result_list+res_tem_int

print(result_list)

my_min=np.min(result_list)
my_max=np.max(result_list)

result_list=(result_list-my_min)/(my_max-my_min)
print(np.max(result_list))

thresholds_all=[Thresholding(r) for r in result_list]

# Group elements into sublists of 5
sublists = [result_list[i:i+5] for i in range(0, len(result_list), 5)]
threshold_sublists=[thresholds_all[i:i+5] for i in range(0, len(thresholds_all), 5)]

# Join each sublist with '|'
scoresNormalized = ['|'.join(map(str, sublist)) for sublist in sublists]
threshold_column=['|'.join(map(str, sublist)) for sublist in threshold_sublists]

print(threshold_column)
queries['scoresNormalized'] = scoresNormalized
queries['Thresholds'] = threshold_column

queries

max_value = queries['scores'].max()
print(max_value)

relations = pd.read_csv('/content/drive/MyDrive/NLP_Dian/train3_2_1.csv')
relations


expected=[]
it=0
for index, row in queries.iterrows():
  d_ids=row.docId
  d_ids_list=d_ids.split('|')
  qid=row.PID

  for d in d_ids_list:
    filtered_rows = relations[(relations['PID'] == qid) & (relations['DocID'] ==d)].Relation.tolist()
    if(len(filtered_rows)!=0):
      r=filtered_rows[0]
      expected.append(r)
      print(r)
    else:
      expected.append(0)

  it=it+1
print(expected)


expected_sublists=[expected[i:i+5] for i in range(0, len(expected), 5)]
expected_column=['|'.join(map(str, sublist)) for sublist in expected_sublists]
queries['expected'] = expected_column
queries


In [ ]:

queries.to_csv('/content/drive/MyDrive/NLP_Dian/queries2_grams.csv', index=False)

def distance(x, y):
  eval_values = [0, 0.1, 0.5, 0.75, 1]
  if (x == y):
    return eval_values[4]
  elif(x - y < -1 ):
    return eval_values[0]
  elif(x - y > 1):
    return eval_values[1]
  elif(x - y == -1):
    return eval_values[2]
  elif(x - y == 1):
    return eval_values[3]

def resScore(res, exp):
  scores = []
  for i in range(len(res)):
    scores.append(distance(res[i], exp[i]))
  return np.mean(scores)

measure=[]
it=0
for index, row in queries.iterrows():
  res=row.Thresholds
  exp=row.expected
  res_list=res.split('|')
  exp_list=exp.split('|')
  res_list=[int(element) for element in res_list]
  exp_list=[int(element) for element in exp_list]
  measure.append(resScore(res_list,exp_list))

print(measure)
queries['measure'] = measure

In [ ]:
queries.describe()